# 00 — Shared Data Audit

Inspect the single frozen five-class split/group contract shared by every version. The optional source audit reads only non-final pixels; this notebook never splits, converts or exports data.

**Inputs:** configs/data.yaml and configs/splits.csv.gz; original BDD sources for the optional audit.

**Outputs:** Read-only membership/support display; no files are written.

**Prerequisites:** Installed project; sources only if AUDIT_SOURCES=True.

**Outline**

- [1. Configuration](#section-1)
- [2. Frozen Membership](#section-2)
- [3. Audit Summary](#section-3)

<a id="section-1"></a>

## 1. Configuration

Every version trains on S20, selects its checkpoint on calibration and reports on development.

In [1]:
import os

import pandas as pd
from IPython.display import display

from vehicle_detection.common import project_root, require_paths
from vehicle_detection.experiment_data import ROLES, audit_sources, shared_splits

ROOT = project_root()
os.chdir(ROOT)  # Run All behaves the same from any kernel working directory.
AUDIT_SOURCES = False
require_paths(shared=ROOT / "configs/data.yaml", splits=ROOT / "configs/splits.csv.gz")
shared, rows = shared_splits(ROOT)

<a id="section-2"></a>

## 2. Frozen Membership

The final role is metadata only. Existing exclusions and duplicate-group unions are retained.

In [2]:
table = pd.DataFrame(rows)
display(
    table.groupby("role").agg(
        images=("id", "size"), groups=("group", "nunique"), s10=("s10", "sum"), s20=("s20", "sum")
    )
)
print("Training subset:", shared["training_subset"], "| Ultralytics val/test roles:", ROLES)
print("Taxonomy:", shared["classes"], "Seed:", shared["seed"])

,images,groups,s10,s20
role,,,,
calibration,6985,4236,0,0
development_validation,13975,8258,0,0
final_test,9730,5836,0,0
training_pool,48903,29615,4890,9781


Training subset: S20 | Ultralytics val/test roles: {'val': 'calibration', 'test': 'development_validation'}
Taxonomy: ['car', 'truck', 'bus', 'motorcycle', 'bicycle'] Seed: 2026


<a id="section-3"></a>

## 3. Audit Summary

Opt in to verify original annotations, source hashes and geometry. No final image reads, headers, hashes, exports or inference occur.

In [3]:
if AUDIT_SOURCES:
    display(audit_sources(ROOT))
else:
    print("Frozen table checked. Original source audit was not requested.")
print("No resplitting; final holdout stays sealed.")

Frozen table checked. Original source audit was not requested.
No resplitting; final holdout stays sealed.
